[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilay47/marginally-realistic/blob/main/cvpr27/notebooks/nb6_e8_time_reversal.ipynb)

# nb6: E8, does each metric detect time reversal of real clips?
Real held-out data only (**no trained models needed**), so it can run as soon as nb0 has finished. Conditions: `real_vs_real` (baseline), `time_reversed` (video reversed, gyro reversed and negated, torque reversed), `video_reversed_only`. Reports the video metric (FVD) with bootstrap CIs, W1, MMD and the three physics diagnostics, plus per-sample AUROC. The same E8 block is also written by nb4 into results.md. Writes new files `eval_<tag>/e8.json` and `e8.md`.

## 1. GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "nvidia-smi: no output")
if not torch.cuda.is_available():
    raise SystemExit("No GPU: Runtime -> Change runtime type -> A100 GPU, then rerun.")
print("cuda:", torch.cuda.get_device_name(0), "| mem GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

## 2. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
ROOT = "/content/drive/MyDrive/physgen_cvpr27"
os.makedirs(ROOT, exist_ok=True)

## 3. Code

In [ ]:
import os
if os.path.exists("/content/repo"):
    !git -C /content/repo pull
else:
    !git clone https://github.com/nilay47/marginally-realistic.git /content/repo
%cd /content/repo
!git log -1 --format="commit %H%n%s"

## 4. Install

In [ ]:
# torch / torchvision come with Colab. pytorchvideo is the second I3D source for FVD (see videofeat.py).
!pip install -q scipy matplotlib pytorchvideo

## 5. Check nb0 outputs

In [ ]:
import os
for f in ["cache/pendulum_T64_dt0.04_train.npz", "cache/pendulum_T64_dt0.04_heldout.npz", "cache/video_metric.json"]:
    assert os.path.exists(f"{ROOT}/{f}"), f"missing {f}: run nb0_benchmark first"
print("nb0 outputs present")

## 6. Run E8 (use a new `--tag` to rerun; nothing is overwritten)

In [ ]:
!cd /content/repo && python -u -m cvpr27.benchmark --e8 --tag e8